In [ ]:
# %% [markdown]
# # MMM Pipeline — VS Code Interactive / Jupyter cell version
# Run each `# %%` block as a cell (VS Code: click "Run Cell" above each block,
# or Shift+Enter). All variables persist across cells in the same kernel —
# no pickle, no hardcoded paths needed.
#
# Put your CSV in the SAME FOLDER as this file and set DATA_PATH below.
# If the file doesn't exist, synthetic data is generated automatically so
# you can still run everything end-to-end today.

# %%
import os
import pickle
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

DATA_PATH = "altima_sales_synthetic.csv"   # <-- just the filename; looked for in this notebook's folder

# %% [markdown]
# ## Stage 1 — Data prep & EDA

# %%
def generate_synthetic_mmm_data(n_weeks=156, seed=42):
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2021-01-04", periods=n_weeks, freq="W-MON")
    t = np.arange(n_weeks)
    trend = 8000 + 15 * t
    seasonality = 1500 * np.sin(2 * np.pi * t / 52) + 800 * np.sin(4 * np.pi * t / 52)

    tv_spend = np.clip(rng.normal(12000, 4000, n_weeks), 0, None)
    digital_spend = np.clip(rng.normal(6000, 2500, n_weeks), 0, None)
    radio_spend = np.clip(rng.normal(3000, 1500, n_weeks), 0, None)
    print_spend = np.clip(rng.normal(2000, 1000, n_weeks), 0, None)
    price = 24000 + rng.normal(0, 300, n_weeks).cumsum() * 0.05
    promo = rng.binomial(1, 0.15, n_weeks)

    def true_adstock(x, theta):
        out = np.zeros_like(x, dtype=float)
        carry = 0.0
        for i, v in enumerate(x):
            carry = v + theta * carry
            out[i] = carry
        return out

    def true_hill(x, alpha, gamma_frac):
        gamma = gamma_frac * x.max()
        return x**alpha / (x**alpha + gamma**alpha)

    tv_effect = 3.2 * true_hill(true_adstock(tv_spend, 0.6), 2.0, 0.5) * 6000
    dig_effect = 2.5 * true_hill(true_adstock(digital_spend, 0.15), 1.2, 0.4) * 4000
    radio_effect = 1.8 * true_hill(true_adstock(radio_spend, 0.35), 1.5, 0.45) * 2000
    print_effect = 1.1 * true_hill(true_adstock(print_spend, 0.25), 1.3, 0.5) * 1200
    price_effect = -0.15 * (price - price.mean())
    promo_effect = 1800 * promo
    noise = rng.normal(0, 500, n_weeks)

    sales = np.clip(trend + seasonality + tv_effect + dig_effect + radio_effect
                     + print_effect + price_effect + promo_effect + noise, 0, None)

    return pd.DataFrame({"date": dates, "sales": sales, "tv_spend": tv_spend,
                          "digital_spend": digital_spend, "radio_spend": radio_spend,
                          "print_spend": print_spend, "price": price, "promo": promo})


if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH, parse_dates=["date"])
    print(f"Loaded real data from {DATA_PATH}: {df.shape}")
else:
    df = generate_synthetic_mmm_data()
    print(f"No {DATA_PATH} found — using synthetic MMM dataset: {df.shape}")

media_cols = [c for c in df.columns if c.endswith("_spend")]
control_cols = ["price", "promo"]

print(df.describe().round(1))

# %%
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(df["date"], df["sales"], color="#e34948")
axes[0].set_title("Weekly sales (KPI)")
for c in media_cols:
    axes[1].plot(df["date"], df[c], label=c)
axes[1].set_title("Weekly media spend by channel")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

# %%
# Time-based split — never shuffle for time series
n_test = int(len(df) * 0.2)
train_idx = slice(0, len(df) - n_test)
test_idx = slice(len(df) - n_test, len(df))
print(f"Train: {len(df) - n_test} weeks | Test: {n_test} weeks")



In [ ]:
# %% [markdown]
# ## Stage 2 — Baseline OLS (deliberately naive, no adstock/saturation)

# %%
X = df[media_cols + control_cols].copy()
y = df["sales"]

X_vif = sm.add_constant(X)
vif = pd.DataFrame({
    "feature": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
print("=== VIF ===")
print(vif.round(2).to_string(index=False))

baseline_model = sm.OLS(y, sm.add_constant(X)).fit()
print(baseline_model.summary())

baseline_r2 = baseline_model.rsquared
dw = durbin_watson(baseline_model.resid)
print(f"\nBaseline R²: {baseline_r2:.3f} | Durbin-Watson: {dw:.3f} "
      f"(far from 2 = residual autocorrelation, i.e. missing carryover structure)")



In [ ]:
# %% [markdown]
# ## Stage 3 — Adstock + Saturation transforms, then Ridge

# %%
def geometric_adstock(x, theta):
    x = np.asarray(x, dtype=float)
    out = np.zeros_like(x)
    carry = 0.0
    for i, v in enumerate(x):
        carry = v + theta * carry
        out[i] = carry
    return out


def hill_saturation(x, alpha, gamma_frac):
    x = np.asarray(x, dtype=float)
    gamma = gamma_frac * x.max() if x.max() > 0 else 1.0
    return x**alpha / (x**alpha + gamma**alpha + 1e-9)


def transform_channel(spend, theta, alpha, gamma_frac):
    return hill_saturation(geometric_adstock(spend, theta), alpha, gamma_frac)


def mape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return np.mean(np.abs((y_true - y_pred) / np.clip(y_true, 1e-9, None))) * 100


def fit_and_score(params, ridge_alpha=1.0):
    Xt = pd.DataFrame(index=df.index)
    for ch in media_cols:
        theta, alpha, gamma_frac = params[ch]
        Xt[ch] = transform_channel(df[ch].values, theta, alpha, gamma_frac)
    for c in control_cols:
        Xt[c] = df[c].values

    scaler = StandardScaler()
    Xt_scaled = pd.DataFrame(scaler.fit_transform(Xt), columns=Xt.columns, index=Xt.index)
    yv = df["sales"].values

    ridge = Ridge(alpha=ridge_alpha)
    ridge.fit(Xt_scaled.iloc[train_idx], yv[train_idx])
    y_pred_test = ridge.predict(Xt_scaled.iloc[test_idx])

    return mape(yv[test_idx], y_pred_test), ridge, scaler, Xt


THETA_GRID = [0.1, 0.3, 0.5, 0.7]
ALPHA_GRID = [0.5, 1.0, 2.0, 3.0]
GAMMA_GRID = [0.3, 0.5, 0.7, 1.0]

best_params = {ch: (0.3, 1.0, 0.5) for ch in media_cols}
for round_num in range(2):
    for ch in media_cols:
        best_mape, best_combo = np.inf, best_params[ch]
        for theta, alpha, gamma_frac in product(THETA_GRID, ALPHA_GRID, GAMMA_GRID):
            trial = dict(best_params)
            trial[ch] = (theta, alpha, gamma_frac)
            score, *_ = fit_and_score(trial)
            if score < best_mape:
                best_mape, best_combo = score, (theta, alpha, gamma_frac)
        best_params[ch] = best_combo
    print(f"Round {round_num + 1} best holdout MAPE: {best_mape:.2f}%")

print("\nBest hyperparameters:")
for ch, p in best_params.items():
    print(f"  {ch}: theta={p[0]}, alpha={p[1]}, gamma={p[2]}")

# %%
final_mape, final_model, final_scaler, X_final = fit_and_score(best_params)

# Convert standardized coefficients back to real sales-dollar units
effective_coef = pd.Series(final_model.coef_ / final_scaler.scale_, index=X_final.columns)
effective_intercept = final_model.intercept_ - np.sum(
    final_model.coef_ * final_scaler.mean_ / final_scaler.scale_
)

y_all = df["sales"].values
y_pred_all = X_final.values @ effective_coef.values + effective_intercept
r2_all = r2_score(y_all, y_pred_all)

print(f"Holdout MAPE: {final_mape:.2f}%")
print(f"Full-sample R² (adstock+saturation+Ridge): {r2_all:.3f}")
print(f"Baseline OLS R² was: {baseline_r2:.3f}  |  Improvement: {r2_all - baseline_r2:+.3f}")



In [ ]:
# %% [markdown]
# ## Stage 4 — Evaluate & visualize (uses variables already in memory, no pickle needed)

# %%
# 1) Actual vs predicted
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(df["date"], y_all, color="#e34948", label="Actual", linewidth=1.6)
ax.plot(df["date"], y_pred_all, color="#2a78d6", label="Predicted", linewidth=1.6, linestyle="--")
ax.set_title(f"Model fit: actual vs predicted sales (R² = {r2_all:.3f})")
ax.legend()
plt.tight_layout()
plt.show()

# %%
# 2) Decomposition — center price on its own mean (0-spend is a meaningless
# reference for a control that's never actually 0; centering folds the
# mean-level effect into baseline, matching Robyn's own baseline concept)
X_decomp = X_final.copy()
baseline_adjustment = effective_intercept
for c in control_cols:
    if c == "promo":
        continue
    mean_val = X_decomp[c].mean()
    baseline_adjustment += effective_coef[c] * mean_val
    X_decomp[c] = X_decomp[c] - mean_val

contributions = X_decomp.multiply(effective_coef, axis=1)
contribution_totals = contributions.sum(axis=0)
baseline_total = baseline_adjustment * len(df)
grand_total = contribution_totals.sum() + baseline_total

pct_contribution = (contribution_totals / grand_total * 100).sort_values()
baseline_pct = baseline_total / grand_total * 100

fig, ax = plt.subplots(figsize=(9, 5))
labels = list(pct_contribution.index) + ["baseline"]
values = list(pct_contribution.values) + [baseline_pct]
colors = ["#1baf7a" if l in media_cols else "#898781" if l == "baseline" else "#4a3aa7" for l in labels]
ax.barh(labels, values, color=colors)
ax.set_xlabel("% of total sales explained")
ax.set_title("Decomposition: contribution by variable")
for i, v in enumerate(values):
    ax.text(v + 0.3, i, f"{v:.1f}%", va="center", fontsize=9)
plt.tight_layout()
plt.show()

# %%
# 3) Response / saturation curves
fig, ax = plt.subplots(figsize=(9, 6))
palette = {"tv_spend": "#1baf7a", "digital_spend": "#eb6834", "radio_spend": "#2a78d6", "print_spend": "#9085e9"}

for ch in media_cols:
    theta, alpha, gamma_frac = best_params[ch]
    adstocked = geometric_adstock(df[ch].values, theta)
    spend_range = np.linspace(0, adstocked.max() * 1.3, 100)
    sat_curve = hill_saturation(spend_range, alpha, gamma_frac)
    ax.plot(spend_range, sat_curve, label=ch, color=palette.get(ch))
    mean_adstocked = adstocked.mean()
    mean_response = hill_saturation(np.array([mean_adstocked]), alpha, gamma_frac)[0]
    ax.scatter([mean_adstocked], [mean_response], color=palette.get(ch), zorder=5)
    ax.annotate(f"{mean_adstocked/1000:.1f}K", (mean_adstocked, mean_response),
                textcoords="offset points", xytext=(6, 4), fontsize=8, color=palette.get(ch))

ax.set_xlabel("Spend (carryover + immediate)")
ax.set_ylabel("Saturation (0-1 scale)")
ax.set_title("Response curves and mean spend by channel")
ax.legend()
plt.tight_layout()
plt.show()

# %%
# 4) ROI per channel = contribution ($) / raw spend ($)  -- raw, not adstocked,
# to avoid double-counting the adstock inflation factor in the denominator
roi = {ch: contribution_totals[ch] / df[ch].sum() for ch in media_cols}
roi_series = pd.Series(roi).sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
colors_roi = ["#639922" if v >= 1 else "#e34948" for v in roi_series.values]
ax.barh(roi_series.index, roi_series.values, color=colors_roi)
ax.axvline(1.0, color="#898781", linestyle="--", linewidth=1)
ax.set_xlabel("ROI (sales $ per $1 spent)")
ax.set_title("Channel ROI (green = profitable, red = below breakeven)")
for i, v in enumerate(roi_series.values):
    ax.text(v + 0.02, i, f"{v:.2f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

print("\n=== Summary ===")
print(pd.DataFrame({"contribution_%": pct_contribution.reindex(media_cols),
                     "ROI": roi_series.reindex(media_cols)}).round(2))